## Installing Libraries

In [ ]:
# BitsAndBytes is a library for 8-bit optimizers and quantization of large language models. 
# It allows you to train and fine-tune large language models using 8-bit precision, which can significantly reduce memory usage and speed up training.
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

In [ ]:
# imports

import os
import requests
from IPython.display import Markdown, display, update_display
from google.colab import drive
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import getpass

## Connecting to Tesla T4 GPU

In [ ]:
# GPU Information
gpu_info = !nvidia-smi
gpu_info

In [ ]:
# Check the T4 status
gpu_info = '\n'.join(gpu_info)
if 'Tesla T4' in gpu_info:
    print("Success - Connected to a T4")
else:
    print("NOT CONNECTED TO A T4")

## Connecting to HuggingFace

In [ ]:
hf_token = getpass.getpass("Enter HuggingFace Token: ")
login(token=hf_token, add_to_git_credential=True)

## Mount the Drive

In [ ]:
drive.mount('/content/drive')
audio_file_path = '/content/drive/MyDrive/Colab Notebooks/ai-meeting-minutes-generator/denver_extract.mp3'

In [ ]:
# Open the file
audio_file = open(audio_file_path, "rb")

## Transcribe Audio to Text

In [10]:
from transformers import pipeline

pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-medium.en",
    dtype=torch.float16,
    # device="cuda",
    return_timestamps=True
    )

In [ ]:
# Load the audio file using torchaudio
import torchaudio
waveform, sample_rate = torchaudio.load(audio_file)

In [ ]:
# Convert stereo to mono by averaging the channels
if waveform.shape[0] > 1:
    waveform = waveform.mean(dim=0, keepdim=True)

# Resample to 16kHz to make it compatible with the Whisper model
if sample_rate != 16000:
    resampler = torchaudio.transforms.Resample(sample_rate, 16000)
    waveform = resampler(waveform)
    sample_rate = 16000

In [ ]:
results = pipe(waveform)
transcript = results['text']

In [ ]:
Markdown(transcript)

## Analyze & Report

In [ ]:
system_message = """
You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.
"""

user_prompt = f"""
Below is an extract transcript of a Denver council meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
{transcript}
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
  ]

## Quantization Config

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    )

In [ ]:
def generate(model, message, quant=True, max_tokens=80):
    tokenizer = AutoTokenizer.from_pretrained(model)
    tokenizer.pad_token = tokenizer.eos_token 

    input_ids = tokenizer.apply_chat_template(
        message, 
        return_tensors="pt", 
        padding=True, 
        truncation=True)
    
    attention_mask = torch.ones_like(input_ids, dtype=torch.long)
    streamer = TextStreamer(tokenizer)

    if quant:
        model = AutoModelForCausalLM.from_pretrained(model, quantization_config=quant_config)
    else:
        model = AutoModelForCausalLM.from_pretrained(model)
    output = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=max_tokens, streamer=streamer)
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
model = "tiiuae/falcon-3b-instruct"

In [ ]:
generate(model=model, message=messages)